# Solar irradiance forecasting

This notebook runs the modular pipeline: Open-Meteo hourly GHI and weather, feature engineering with solar geometry and clearness, then baselines plus **LightGBM quantile** models for 1 h, 3 h, and 6 h horizons.

**Stochastic view:** clearness and cloud-driven fluctuations act like multiplicative noise on clear-sky curves; quantile boosting captures heteroscedastic spread (wider intervals around variable cloud regimes) without a full physical stochastic model.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve() / "labs" / "solar_irradiance_forecasting"
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.pipeline import run_solar_irradiance_pipeline
from src.inference import load_validation_predictions
from src.visualization import plot_forecast_intervals, plot_metrics_heatmap, plot_residuals_by_hour

artifacts = run_solar_irradiance_pipeline(PROJECT_ROOT)
artifacts.model_artifacts.metrics.round(4)

In [ ]:
preds = load_validation_predictions(artifacts.predictions_path)
plot_forecast_intervals(preds, horizon_h=1)
plot_residuals_by_hour(preds, horizon_h=1)
plot_metrics_heatmap(artifacts.model_artifacts.metrics)

## Metrics interpretation

- **Persistence** uses current GHI as the forecast; strong at 1 h, weaker at 6 h.
- **Hourly median** uses the training-set median GHI for each UTC hour; a crude seasonality prior.
- **LightGBM quantile** rows include pinball losses for 0.1 / 0.5 / 0.9 and **interval_coverage_80**: share of observations inside the 10–90% band.